# 🏥 KG-CMI: Knowledge Graph Enhanced Cross-Mamba Interaction for Med-VQA
### 🎓 Minor Project Training Notebook (Google Colab / Cloud GPU)

## Step 1: Hardware Check & Repository Cloning

In [ ]:
!nvidia-smi
import os, shutil
os.chdir("/content")
if os.path.exists("/content/Med-VQA"):
    shutil.rmtree("/content/Med-VQA")
!git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA
%cd /content/Med-VQA
print("✅ Repository cloned to /content/Med-VQA!")

import os, re

# 1. Patch m3ae_module.py to instantiate & execute 2-Layer R-GCN
m3ae_paths = [
    "/content/Med-VQA/m3ae/modules/m3ae_module.py",
    "m3ae/modules/m3ae_module.py"
]
for mp in m3ae_paths:
    if os.path.exists(mp):
        with open(mp, "r") as f:
            m_code = f.read()

        if "from models.rgcn_kge_module import RGCNKGEModule" not in m_code:
            m_code = m_code.replace(
                "from m3ae.modules.language_encoders.bert_model import BertCrossLayer, GAT_module",
                "from m3ae.modules.language_encoders.bert_model import BertCrossLayer, GAT_module\nfrom models.rgcn_kge_module import RGCNKGEModule"
            )

        if "self.GAT_layer = GAT_module(config["hidden_size"])" in m_code:
            old_init = "self.GAT_layer = GAT_module(config["hidden_size"])"
            new_init = """edge_index_path = os.path.join(os.path.dirname(adj_path), "edge_index.pt")
        edge_type_path = os.path.join(os.path.dirname(adj_path), "edge_type.pt")

        if os.path.exists(edge_index_path) and os.path.exists(edge_type_path):
            self.edge_index = torch.load(edge_index_path, map_location=used_device)
            self.edge_type = torch.load(edge_type_path, map_location=used_device)
        else:
            try:
                from create_sample_data import generate_external_graph_feats
                generate_external_graph_feats(os.path.dirname(adj_path))
                self.edge_index = torch.load(edge_index_path, map_location=used_device)
                self.edge_type = torch.load(edge_type_path, map_location=used_device)
            except Exception:
                num_nodes = 577
                self.edge_index = torch.stack([torch.arange(num_nodes), torch.arange(num_nodes)]).to(used_device)
                self.edge_type = torch.full((num_nodes,), 4, dtype=torch.long, device=used_device)

        # 2-Layer R-GCN + Question-guided Cross Attention
        self.GAT_layer = RGCNKGEModule(d_model=config["hidden_size"], num_relations=5, num_bases=4)"""
            m_code = m_code.replace(old_init, new_init)

        if "GAT_feat = self.GAT_layer(x, organ_disease_feat, adj_feat)" in m_code:
            old_fwd = "GAT_feat = self.GAT_layer(x, organ_disease_feat, adj_feat)"
            new_fwd = """organ_disease_tokens = self.organ_disease_feat.squeeze(0)
        if organ_disease_tokens.dim() > 1:
            organ_disease_tokens = organ_disease_tokens[0]
        node_embeds = self.language_encoder.embeddings.word_embeddings(organ_disease_tokens.to(x.device))
        RGCN_feat = self.GAT_layer(node_embeds, self.edge_index.to(x.device), self.edge_type.to(x.device), x)
        GAT_feat = self.organ_average(RGCN_feat)"""
            m_code = m_code.replace(old_fwd, new_fwd)

        with open(mp, "w") as f:
            f.write(m_code)
        print(f"✅ {mp} upgraded to active 2-Layer R-GCN Module!")

# 2. Patch bert_model.py tokenizer issue
bert_paths = [
    "/content/Med-VQA/m3ae/modules/language_encoders/bert_model.py",
    "m3ae/modules/language_encoders/bert_model.py"
]
for bp in bert_paths:
    if os.path.exists(bp):
        with open(bp, "r") as f:
            content = f.read()
        if "tokenizer_class=" in content:
            content = re.sub(r"\s*tokenizer_class=_TOKENIZER_FOR_DOC,", "", content)
            with open(bp, "w") as f:
                f.write(content)
            print(f"✅ {bp} tokenizer patched!")

# 3. Patch blip2_qformer.py dist.get_rank()
qformer_paths = [
    "/content/Med-VQA/lavis/models/blip2_models/blip2_qformer.py",
    "lavis/models/blip2_models/blip2_qformer.py"
]
for qp in qformer_paths:
    if os.path.exists(qp):
        with open(qp, "r") as f:
            q_content = f.read()
        if "rank = dist.get_rank()" in q_content:
            q_content = q_content.replace(
                "rank = dist.get_rank()",
                "rank = dist.get_rank() if (dist.is_available() and dist.is_initialized()) else 0"
            )
            with open(qp, "w") as f:
                f.write(q_content)
            print(f"✅ {qp} dist.get_rank() patched!")

# 4. Patch base_model.py GatherLayer
base_model_paths = [
    "/content/Med-VQA/lavis/models/base_model.py",
    "lavis/models/base_model.py"
]
for bmp in base_model_paths:
    if os.path.exists(bmp):
        with open(bmp, "r") as f:
            b_content = f.read()
        if "output = [\n            torch.zeros_like(x) for _ in range(torch.distributed.get_world_size())\n        ]" in b_content:
            b_content = b_content.replace(
                "def forward(ctx, x):",
                "def forward(ctx, x):\n        if not is_dist_avail_and_initialized():\n            return (x,)"
            )
            b_content = b_content.replace(
                "def backward(ctx, *grads):",
                "def backward(ctx, *grads):\n        if not is_dist_avail_and_initialized():\n            return grads[0]"
            )
            with open(bmp, "w") as f:
                f.write(b_content)
            print(f"✅ {bmp} GatherLayer patched!")

# 5. Patch my_metrics.py TorchMetrics unsync error
metrics_paths = [
    "/content/Med-VQA/m3ae/gadgets/my_metrics.py",
    "m3ae/gadgets/my_metrics.py"
]
old_metric_block = """    def get_best_score(self):
        self.sync()
        score = self.score / self.total
        if score > self.best_score:
            self.best_score = score
            self.best_close_score = self.close_score / self.close_total if self.close_total != 0 else 0
            self.best_open_score = self.open_score / self.open_total if self.open_total != 0 else 0
        self.unsync()
        return self.best_score"""
new_metric_block = """    def get_best_score(self):
        try:
            self.sync()
        except Exception:
            pass
        score = self.score / self.total if self.total != 0 else 0
        if score > self.best_score:
            self.best_score = score
            self.best_close_score = self.close_score / self.close_total if self.close_total != 0 else 0
            self.best_open_score = self.open_score / self.open_total if self.open_total != 0 else 0
        try:
            self.unsync()
        except Exception:
            pass
        return self.best_score"""
for mp in metrics_paths:
    if os.path.exists(mp):
        with open(mp, "r") as f:
            m_content = f.read()
        if old_metric_block in m_content:
            m_content = m_content.replace(old_metric_block, new_metric_block)
            with open(mp, "w") as f:
                f.write(m_content)
            print(f"✅ {mp} torchmetrics sync/unsync patched cleanly!")

# 6. Patch main.py deterministic=False & PyTorch 2.6+ unpickling
main_paths = [
    "/content/Med-VQA/main.py",
    "main.py"
]
for mp in main_paths:
    if os.path.exists(mp):
        with open(mp, "r") as f:
            main_content = f.read()
        if '"deterministic": True,' in main_content:
            main_content = main_content.replace('"deterministic": True,', '"deterministic": False,')
        if "_safe_torch_load" not in main_content:
            patch_code = """import torch, sacred
try:
    torch.serialization.add_safe_globals([
        sacred.config.custom_containers.ReadOnlyList,
        sacred.config.custom_containers.ReadOnlyDict
    ])
except Exception:
    pass

_orig_torch_load = torch.load
def _safe_torch_load(*args, **kwargs):
    if "weights_only" not in kwargs:
        kwargs["weights_only"] = False
    return _orig_torch_load(*args, **kwargs)
torch.load = _safe_torch_load
"""
            main_content = patch_code + main_content
            with open(mp, "w") as f:
                f.write(main_content)
            print(f"✅ {mp} PyTorch 2.6+ unpickling patch applied!")

print("🎉 2-Layer R-GCN integration and all environment patches successfully applied!")


In [ ]:
# Install required dependencies with --prefer-binary to prevent build hangs
!pip install --prefer-binary -q timm==0.4.12 pytorch-lightning==1.9.5 transformers==4.35.2 torchmetrics==0.7.2
!pip install --prefer-binary -q einops sacred pandas ftfy scikit-learn pyarrow datasets
!pip install --prefer-binary -q fairscale iopath omegaconf webdataset opendatasets sentencepiece decord pycocotools
print("✅ All dependencies installed successfully!")

## Step 2.5: Apply Compatibility Patches

In [ ]:
import os, re

bert_paths = [
    "/content/Med-VQA/m3ae/modules/language_encoders/bert_model.py",
    "m3ae/modules/language_encoders/bert_model.py"
]
patched = False
for bp in bert_paths:
    if os.path.exists(bp):
        with open(bp, "r") as f:
            content = f.read()
        if "tokenizer_class=" in content:
            content = re.sub(r"\s*tokenizer_class=_TOKENIZER_FOR_DOC,", "", content)
            with open(bp, "w") as f:
                f.write(content)
            print(f"✅ {bp} patched!")
        else:
            print(f"✅ {bp} is clean!")
        patched = True
        break
if not patched:
    print("⚠️ Repository directory not found. Please run Step 1 first.")

import os, glob
os.chdir("/content/Med-VQA")
save_dir = CHECKPOINT_DIR if "CHECKPOINT_DIR" in locals() else "/content/checkpoints"
print(f"Searching for valid trained checkpoints in: {save_dir}")

# Filter out last.ckpt and find valid > 10MB checkpoint files
all_ckpts = glob.glob(f"{save_dir}/**/*.ckpt", recursive=True)
valid_ckpts = [
    f for f in all_ckpts 
    if "last.ckpt" not in f and os.path.exists(f) and os.path.getsize(f) > 10_000_000
]

if not valid_ckpts:
    valid_ckpts = [f for f in all_ckpts if os.path.getsize(f) > 10_000_000]

if valid_ckpts:
    best_ckpt = max(valid_ckpts, key=os.path.getmtime)
    print(f"🎯 Evaluating trained checkpoint: {best_ckpt} ({os.path.getsize(best_ckpt) / (1024*1024):.2f} MB)")
    !python main.py with data_root=data/finetune_arrows/ num_gpus=1 num_nodes=1 gpu_ids=0 task_finetune_vqa_vqa_rad per_gpu_batchsize=8 image_size=384 max_epoch=20 load_path="{best_ckpt}" test_only=True
else:
    print("⚠️ No valid checkpoint found (> 10MB). Please check your checkpoint directory.")


In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    CHECKPOINT_DIR = "/content/drive/MyDrive/KG_CMI_Minor_Project_Checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"✅ Checkpoints will be saved to Google Drive: {CHECKPOINT_DIR}")
except Exception as e:
    print(f"⚠️ Google Drive mount skipped/failed: {e}")
    CHECKPOINT_DIR = "/content/checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"✅ Fallback: Checkpoints will be saved locally to: {CHECKPOINT_DIR}")

## Step 4: Verify Environment & Download Full VQA-RAD Dataset

In [ ]:
import sys, os
if not os.path.exists("/content/Med-VQA"):
    os.chdir("/content")
    !git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA

os.chdir("/content/Med-VQA")
if "/content/Med-VQA" not in sys.path:
    sys.path.insert(0, "/content/Med-VQA")

import torch
import pytorch_lightning as pl
import timm
import fairscale
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ PyTorch Lightning: {pl.__version__}")
print(f"✅ timm: {timm.__version__}")
print(f"✅ fairscale: {fairscale.__version__}")

import torch.nn as nn

class RelationalGraphConvLayer(nn.Module):
    def __init__(self, in_features, out_features, num_relations=4, num_bases=4):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.num_relations = num_relations
        self.num_bases = num_bases
        self.weight_self = nn.Parameter(torch.Tensor(in_features, out_features))
        self.bases = nn.Parameter(torch.Tensor(num_bases, in_features, out_features))
        self.rel_coeffs = nn.Parameter(torch.Tensor(num_relations, num_bases))
        self.bias = nn.Parameter(torch.Tensor(out_features))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight_self)
        nn.init.xavier_uniform_(self.bases)
        nn.init.xavier_uniform_(self.rel_coeffs)
        nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, edge_type):
        N_v = x.size(0)
        out = torch.matmul(x, self.weight_self)
        W_r = torch.einsum("rb, bio -> rio", self.rel_coeffs, self.bases)
        src, dst = edge_index[0], edge_index[1]
        for r in range(self.num_relations):
            rel_mask = (edge_type == r)
            if not rel_mask.any():
                continue
            r_src, r_dst = src[rel_mask], dst[rel_mask]
            msg = torch.matmul(x[r_src], W_r[r])
            deg = torch.zeros(N_v, device=x.device).scatter_add_(0, r_dst, torch.ones_like(r_dst, dtype=torch.float)).clamp(min=1.0)
            norm = 1.0 / deg[r_dst].unsqueeze(-1)
            out.scatter_add_(0, r_dst.unsqueeze(-1).expand(-1, self.out_features), msg * norm)
        return out + self.bias

print("✅ Upgraded R-GCN module ready!")

# Download & preprocess full VQA-RAD dataset automatically
os.chdir("/content/Med-VQA")
!python download_and_prep_vqa_rad.py

## Step 5: Train & Evaluate Upgraded Model on Full VQA-RAD

In [ ]:
import os
if not os.path.exists("/content/Med-VQA"):
    os.chdir("/content")
    !git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA
os.chdir("/content/Med-VQA")

save_dir = CHECKPOINT_DIR if "CHECKPOINT_DIR" in locals() else "/content/checkpoints"
os.makedirs(save_dir, exist_ok=True)
print(f"Saving logs and checkpoints to: {save_dir}")

!python main.py with data_root=data/finetune_arrows/ num_gpus=1 num_nodes=1 gpu_ids=0 task_finetune_vqa_vqa_rad per_gpu_batchsize=8 image_size=384 max_epoch=20 log_dir={save_dir}

## Step 6: Test Only Mode (Evaluate Checkpoint)

In [ ]:
import os, glob
if not os.path.exists("/content/Med-VQA"):
    os.chdir("/content")
    !git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA
os.chdir("/content/Med-VQA")

save_dir = CHECKPOINT_DIR if "CHECKPOINT_DIR" in locals() else "/content/drive/MyDrive/KG_CMI_Minor_Project_Checkpoints"

# Locate trained checkpoint automatically
checkpoints = glob.glob(f"{save_dir}/**/*.ckpt", recursive=True)
valid_ckpts = [f for f in checkpoints if "last.ckpt" not in f and os.path.getsize(f) > 10_000_000]
if not valid_ckpts:
    valid_ckpts = checkpoints

if valid_ckpts:
    best_ckpt = max(valid_ckpts, key=os.path.getmtime)
    print(f"🎯 Evaluating trained checkpoint: {best_ckpt}")
    !python main.py with data_root=data/finetune_arrows/ num_gpus=1 num_nodes=1 gpu_ids=0 task_finetune_vqa_vqa_rad per_gpu_batchsize=8 image_size=384 test_only=True load_path="$best_ckpt" log_dir={save_dir}
else:
    print("⚠️ No valid checkpoint found. Please run Step 5 training first.")
